In [37]:
import os
import xml.etree.ElementTree as ET
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms as transforms


class PedestrianDataset(Dataset):
    def __init__(self, frames_dir, labels_dir, transform=None):
        self.frames_dir = frames_dir
        self.labels_dir = labels_dir
        self.transform = transform
        
        # Match PNGs with their corresponding XML files dynamically
        self.filenames = []
        for f in sorted(os.listdir(frames_dir)):
            if f.endswith('.png'):
                base_name = os.path.splitext(f)[0]
                xml_name = f"{base_name}.xml"
                if os.path.exists(os.path.join(labels_dir, xml_name)):
                    self.filenames.append(base_name)
                    
        print(f"Matched {len(self.filenames)} image/label pairs successfully.")

    def __len__(self):
        return len(self.filenames)

    def parse_xml(self, xml_path, idx):
        tree = ET.parse(xml_path)
        root = tree.getroot()
        
        boxes = []
        labels = []
        areas = []
        
        for obj in root.findall('object'):
            name = obj.find('name').text
            
            # CRITICAL: Match 'person' from your XML file
            if name == 'person':
                labels.append(1)  # Class 1 = Person (Class 0 is background)
                
                # Extract Bounding Box
                bndbox = obj.find('bndbox')
                xmin = float(bndbox.find('xmin').text)
                ymin = float(bndbox.find('ymin').text)
                xmax = float(bndbox.find('xmax').text)
                ymax = float(bndbox.find('ymax').text)
                
                boxes.append([xmin, ymin, xmax, ymax])
                
                # Calculate box area (width * height)
                area = (xmax - xmin) * (ymax - ymin)
                areas.append(area)
        
        # Convert everything to standard PyTorch Tensors
        target = {}
        target["boxes"] = torch.as_tensor(boxes, dtype=torch.float32)
        target["labels"] = torch.as_tensor(labels, dtype=torch.int64)
        target["image_id"] = torch.tensor([idx], dtype=torch.int64)
        target["area"] = torch.as_tensor(areas, dtype=torch.float32)
        target["iscrowd"] = torch.zeros((len(labels),), dtype=torch.int64) # Assuming no crowd tags
        
        return target

    def __getitem__(self, idx):
        base_name = self.filenames[idx]
        
        img_path = os.path.join(self.frames_dir, f"{base_name}.png")
        xml_path = os.path.join(self.labels_dir, f"{base_name}.xml")
        
        # Your XML lists depth=1 (Grayscale). 
        # Note: Most torchvision detection models still expect 3 channels (RGB).
        # Convert to "RGB" (replicates the grayscale channel 3 times) to avoid model errors.
        image = Image.open(img_path).convert("RGB") 
        
        target = self.parse_xml(xml_path, idx)
        
        if self.transform:
            image = self.transform(image)
            
        return image, target

In [38]:
# Define your image preprocessing
# image_transforms = transforms.Compose([
#     transforms.Resize((256, 256)),  # Change to your model's input size
#     transforms.ToTensor(),
#     transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) # Standard ImageNet normalization
# ])

# Define your label preprocessing (e.g., if it's a segmentation mask)
label_transforms = transforms.Compose([
    transforms.Resize((256, 256), interpolation=transforms.InterpolationMode.NEAREST), # Nearest neighbor keeps mask values crisp
    transforms.ToTensor() # Converts to a tensor [0.0, 1.0] or class indices
])

# Define the absolute or relative paths to your folders
# Double check the 'Pedestrain' vs 'Pedestrian' spelling here based on your actual system!
TRAIN_DIR = "data/train" 
FRAMES_DIR = os.path.join(TRAIN_DIR, "Pedestrian frame")
LABELS_DIR = os.path.join(TRAIN_DIR, "Pedestrian label") 

# Create the dataset instance
pedestrian_dataset = PedestrianDataset(
    frames_dir=FRAMES_DIR,
    labels_dir=LABELS_DIR
)

# Create the DataLoader
train_loader = DataLoader(
    dataset=pedestrian_dataset,
    batch_size=1024,        # Adjust based on your GPU/CPU memory
    shuffle=True,        # Shuffles the data every epoch
    num_workers=8        # Speeds up data loading
)

Matched 4119 image/label pairs successfully.


In [39]:
import torch
import torch.nn as nn
import snntorch as snn
from snntorch import surrogate

class SpikingPedestrianDetector(nn.Module):
    def __init__(self, beta=0.9, num_steps=3):
        super().__init__()
        self.num_steps = num_steps # Number of time steps to simulate spikes
        
        # Spike gradient surrogate function (allows backpropagation over discrete spikes)
        spike_grad = surrogate.fast_sigmoid(slope=25)
        
        # 1. Spiking Convolutional Backbone
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, stride=2, padding=1) # Downsample 256 -> 128
        self.lif1 = snn.Leaky(beta=beta, spike_grad=spike_grad)
        
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, stride=2, padding=1) # Downsample 128 -> 64
        self.lif2 = snn.Leaky(beta=beta, spike_grad=spike_grad)
        
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1) # Downsample 64 -> 32
        self.lif3 = snn.Leaky(beta=beta, spike_grad=spike_grad)
        
        # 2. Dense Predictor Heads (Reads flattened spike features)
        # 64 channels * 32 * 32 spatial grid = 65,536 features
        self.flatten = nn.Flatten()
        
        # Bounding Box Regressor (predicts 4 coordinates: xmin, ymin, xmax, ymax)
        self.bbox_head = nn.Linear(64 * 32 * 32, 4) 
        
        # Classification Head (predicts probability of 'person' presence)
        self.cls_head = nn.Linear(64 * 32 * 32, 1) 

    def forward(self, x):
        # Initialize membrane potentials for LIF neurons
        mem1 = self.lif1.init_leaky()
        mem2 = self.lif2.init_leaky()
        mem3 = self.lif3.init_leaky()
        
        # Accumulators for spikes/outputs over time steps
        spk3_sum = torch.zeros((x.size(0), 64, 32, 32), device=x.device)
        
        # Static input emulation: feed the same image across all time steps
        for step in range(self.num_steps):
            cur1 = self.conv1(x)
            spk1, mem1 = self.lif1(cur1, mem1)
            
            cur2 = self.conv2(spk1)
            spk2, mem2 = self.lif2(cur2, mem2)
            
            cur3 = self.conv3(spk2)
            spk3, mem3 = self.lif3(cur3, mem3)
            
            spk3_sum += spk3 # Accumulate spikes over time
            
        # Average spike feature map across time
        feat = spk3_sum / self.num_steps
        feat_flat = self.flatten(feat)
        
        # Final predictions
        pred_boxes = self.bbox_head(feat_flat)
        pred_logits = self.cls_head(feat_flat)
        
        return pred_boxes, pred_logits

In [56]:
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

# 1. Transforms to handle 256x256 conversion
image_transforms = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
])

# 2. Instantiate Dataset and Loader (Assuming 1 image at a time for simple regression)
# Use the PedestrianPascalVOCDataset class we defined in the previous prompt
dataset = PedestrianDataset(FRAMES_DIR, LABELS_DIR, transform=image_transforms)
# train_loader = DataLoader(dataset, batch_size=1, shuffle=True)

def collate_fn(batch):
    return tuple(zip(*batch))

train_loader = DataLoader(
    dataset, 
    batch_size=32,       # <--- Increase this (e.g., 16, 32, 64) to fill VRAM
    shuffle=True, 
    num_workers=0,       # Uses CPU multi-threading to feed the GPU faster
    pin_memory=True,     # <--- Crucial: Pins tensors to system RAM for faster GPU transfer
    collate_fn=collate_fn
)

# 3. Model, Optimizer, and Loss functions
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SpikingPedestrianDetector(beta=0.9, num_steps=3).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# Losses
bbox_loss_fn = nn.MSELoss() # For coordinates bounding boxes
cls_loss_fn = nn.BCEWithLogitsLoss() # For object classification

Matched 4119 image/label pairs successfully.


In [57]:
device

device(type='cuda')

In [ ]:
num_epochs = 5

for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0
    
    for images, targets in train_loader:
        # 1. Skip empty frames / unpack safely depending on batch layout
        # if targets is a single dict (batch_size=1), wrap it in a tuple to match batch logic
        if isinstance(targets, dict):
            targets = (targets,)
            
        # Check if the first image in the batch has a box (for this basic regression setup)
        if targets[0]["boxes"].size(0) == 0:
            continue 

        # 2. Process image tensor dimensions using the type check fix from earlier
        if isinstance(images, (tuple, list)):
            images = torch.stack(images).to(device)
        elif isinstance(images, torch.Tensor):
            if images.dim() == 3:
                images = images.unsqueeze(0) 
            images = images.to(device)

        # 3. Forward Pass through the Spiking Neural Network
        pred_boxes, pred_logits = model(images)
        
        # 4. Compute Loss across the batch
        # For this basic regression setup, we will gather targets from the batch items
        batch_loss = 0
        orig_w, orig_h = 346, 260 # Dimensions from your a130.xml
        
        for i in range(len(targets)):
            # --- THE FIX HAPPENS HERE ---
            # Look inside target 'i' of the tuple, then fetch the first box tensor
            if targets[i]["boxes"].size(0) == 0:
                continue
                
            true_box = targets[i]["boxes"][0].to(device) # Fetch first box coordinates
            
            # Normalize coordinates [0, 1]
            true_box_scaled = torch.tensor([
                true_box[0] / orig_w,
                true_box[1] / orig_h,
                true_box[2] / orig_w,
                true_box[3] / orig_h
            ], device=device)
            
            true_cls = torch.tensor([1.0], device=device)
            
            # Calculate loss against model outputs for index i
            loss_box = bbox_loss_fn(pred_boxes[i], true_box_scaled)
            loss_cls = cls_loss_fn(pred_logits[i], true_cls)
            
            batch_loss += (loss_box + loss_cls)
            
        # Average the loss over the batch size
        total_loss = batch_loss / len(targets)
        
        # 5. Optimization step
        optimizer.zero_grad()
        total_loss.backward()
        optimizer.step()
        
        epoch_loss += total_loss.item()
        
    print(f"Epoch [{epoch+1}/{num_epochs}] | Total Loss: {epoch_loss:.4f}")

TypeError: tuple indices must be integers or slices, not str